# CENG468 Veri Madenciliği · Hafta 9
## MovieLens: yeni veri setimizi tanıyalım
Öğr. Gör. Dr. Duygu TOPALOĞLU · 2026-2027 Güz

---
### Bu derste nasıl çalışacağız?
İşaretler önceki haftalardaki gibi:

| İşaret | Ne yapıyorsun? |
|---|---|
| 🔮 **Tahmin et** | Hücreyi çalıştırmadan önce çıktıyı tahmin et ve yaz. Sonra çalıştır ve karşılaştır. |
| 🔧 **Değiştir** | Çalışan kodu küçük bir farkla uyarla. |
| ✍️ **Kendin yaz** | Boş hücreye kodu kendin yaz. Kopyala-yapıştır yok. |
| 🐞 **Hata avı** | Kod bilerek hatalı. Hata mesajını oku, nedenini bul, düzelt. |
| 🕵️ **İz sür** | Tuhaf bir sonucun nedenini veride bul. |

**İpuçları:** Bazı görevlerin altında kapalı 💡 kutular var. **Önce en az bir kez kendin dene**, takılırsan yalnızca bir sonraki ipucunu aç.

Şimdiye kadar **Online Retail II** ile çalıştık: faturalar ve ürünler. Ara sınavdan sonra öneri sistemlerine geçiyoruz ve ikinci bir veri setimiz oluyor: **MovieLens**. Kullanıcılar filmlere 0,5 ile 5 yıldız arasında puan veriyor.

Bugünün yolu: **dört tabloyu tanı → puanlara, kullanıcılara, filmlere grafikle bak → tuhaflıkları bul → temizle → kaydet.**

---
## Bölüm 0: Hazırlık (5 dk)
Veri dosyaları repoda yok. Henüz indirmediysen `Veri/movielens/BENIOKU.md` dosyasındaki adımları izle (zip yaklaşık 1 MB).

Aşağıdaki hücre dört dosyayı okur. Yalnızca çalıştır.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Grafiklerde kullanacağımız renkler (H06 ile aynı)
YESIL, KIRMIZI, MAVI, GRI, TURUNCU, MOR = "#2e7d32", "#c62828", "#1565c0", "#9e9e9e", "#ef6c00", "#6a1b9a"

VERI = os.path.join("Veri", "movielens", "ml-latest-small")
if not os.path.exists(os.path.join(VERI, "movies.csv")):           # zip bir kat fazla açıldıysa
    VERI = os.path.join(VERI, "ml-latest-small")

if os.path.exists(os.path.join(VERI, "movies.csv")):
    filmler   = pd.read_csv(os.path.join(VERI, "movies.csv"))
    puanlar   = pd.read_csv(os.path.join(VERI, "ratings.csv"))
    etiketler = pd.read_csv(os.path.join(VERI, "tags.csv"))
    baglar    = pd.read_csv(os.path.join(VERI, "links.csv"))
    print("filmler  :", filmler.shape)
    print("puanlar  :", puanlar.shape)
    print("etiketler:", etiketler.shape)
    print("baglar   :", baglar.shape)
else:
    print("Veri dosyası bulunamadı:", os.path.abspath(VERI))
    print("Veri/movielens/BENIOKU.md dosyasındaki adımları izle.")

Aşağıdaki hücre bugün kullanacağımız iki yardımcı fonksiyonu tanımlar. `adim` fonksiyonunu H05'ten hatırlıyorsun. **İçlerini şimdilik okumana gerek yok**; yalnızca çalıştır.

In [ ]:
def adim(veri, maske, aciklama):
    """Maskesi True olan satırları tutar; kaç satır çıktığını yazar."""
    sonra = veri[maske]
    print(f"{aciklama:<38} çıkan: {len(veri) - len(sonra):>6,}   kalan: {len(sonra):>7,}")
    return sonra


def kontrol_temiz(f, p, e):
    """Temizlenmiş tabloları sınıf kararlarına göre denetler."""
    testler = {
        "Aynı adla iki film yok":                 not f["title"].duplicated().any(),
        "Başlıklarda baş/son boşluk yok":         (f["title"] == f["title"].str.strip()).all(),
        "'(no genres listed)' kalmadı":           not (f["genres"] == "(no genres listed)").any(),
        "Her puan bilinen bir filme ait":         p["movieId"].isin(f["movieId"]).all(),
        "Aynı kullanıcı-film çifti tek":          not p.duplicated(["userId", "movieId"]).any(),
        "Puanlar 0,5 ile 5 arasında":             p["rating"].between(0.5, 5).all(),
        "Puan zamanı tarih tipinde":              pd.api.types.is_datetime64_any_dtype(p["zaman"]),
        "Etiketler küçük harf, boşluksuz":        (e["tag"] == e["tag"].str.lower().str.strip()).all(),
    }
    for ad, sonuc in testler.items():
        print("✅" if sonuc else "❌", ad)

---
## Bölüm 1: Dört tablo, bir veri seti (15 dk)
Online Retail tek bir tabloydu. MovieLens ise **dört tablodan** oluşur. Tablolar birbirine ortak sütunlarla (**anahtar**) bağlanır:

```
            ┌──────────────┐
            │   filmler    │  movieId, title, genres
            └──────┬───────┘
                   │ movieId
     ┌─────────────┼──────────────┐
     │             │              │
┌────┴─────┐  ┌────┴─────┐  ┌─────┴────┐
│ puanlar  │  │etiketler │  │  baglar  │
│ userId   │  │ userId   │  │ imdbId   │
│ movieId  │  │ movieId  │  │ tmdbId   │
│ rating   │  │ tag      │  └──────────┘
│ timestamp│  │ timestamp│
└──────────┘  └──────────┘
```

Kullanıcılar için ayrı bir tablo yok: kullanıcıyı yalnızca **numarası** (`userId`) ve **verdiği puanlar** ile tanıyoruz. Yaş, cinsiyet, şehir gibi bilgiler yok.

### 1.1 İlk bakış

In [ ]:
filmler.head()

In [ ]:
puanlar.head()

In [ ]:
etiketler.head()

In [ ]:
baglar.head()

**🔮 Düşün:** `puanlar` tablosundaki `timestamp` sütununda `964982703` gibi sayılar var. Bu bir tarih mi? Nasıl okunur?

*Cevabım:* ...

<details><summary>💡 İpucu 1</summary>

Bu bir **Unix zaman damgası**: 1 Ocak 1970'ten bu yana geçen **saniye** sayısı. Bölüm 5'te tarihe çevireceğiz.

</details>

In [ ]:
puanlar.info()

**✍️ Tabloyu doldur:** Her sütun ne tür bir değişken?

| Tablo | Sütun | Anlamı | Tür (sayısal / kategorik / metin / zaman) | Ölçek (nominal / ordinal / aralık / oran) |
|---|---|---|---|---|
| filmler | movieId | | | |
| filmler | title | | | |
| filmler | genres | | | |
| puanlar | userId | | | |
| puanlar | rating | | | |
| puanlar | timestamp | | | |
| etiketler | tag | | | |

**🔮 Düşün:** `rating` sayı olarak okunmuş. 4 yıldız, 2 yıldızın "iki katı kadar beğenmek" mi? Ölçeği oran mı, ordinal mi?

### 1.2 Kaç kullanıcı, kaç film?
**🔮 Tahmin et:** `puanlar` tablosunda 100.836 satır var. Kaç **farklı** kullanıcı ve kaç **farklı** film vardır?

*Tahminim:* ...

In [ ]:
n_kullanici = puanlar["userId"].nunique()
n_film = puanlar["movieId"].nunique()
print("Farklı kullanıcı      :", n_kullanici)
print("Puan almış farklı film:", n_film)
print("movies.csv'deki film  :", len(filmler))

**🕵️ İz sür:** `movies.csv`'de 9.742 film var ama puan almış film sayısı daha az. Puan almamış filmler hangileri? Kaç tane?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

Bir sütundaki değerlerin başka bir sütunda olup olmadığını `.isin()` ile sorarsın. Tersini almak için `~`.

</details>

<details><summary>💡 İpucu 2</summary>

`filmler[~filmler["movieId"].isin(puanlar["movieId"])]`

</details>

### 1.3 Eksik değer ve tekrar var mı?

In [ ]:
for ad, tablo in [("filmler", filmler), ("puanlar", puanlar), ("etiketler", etiketler), ("baglar", baglar)]:
    print(f"{ad:<10} eksik: {tablo.isna().sum().to_dict()}   tekrarlı satır: {tablo.duplicated().sum()}")

Tablolar ilk bakışta tertemiz görünüyor: neredeyse hiç boş hücre, hiç tekrarlı satır yok. Online Retail'deki gibi iptal faturası, negatif adet yok.

**🔮 Düşün:** Bu, verinin temiz olduğu anlamına mı geliyor? `isna()` ve `duplicated()` hangi tür sorunları **göremez**?

*Cevabım:* ...

Cevabı Bölüm 6'da göreceğiz. Önce veriyi grafiklerle tanıyalım.

---
## Bölüm 2: Puanlar nasıl dağılıyor? (10 dk)
**🔮 Tahmin et:** İnsanlar en çok hangi puanı veriyor? Tam yıldızlar (1, 2, 3, 4, 5) mı daha sık, buçuklu yıldızlar (0,5, 1,5, ...) mı? Ortalama puan kaç civarındadır?

*Tahminim:* ...

In [ ]:
dagilim = puanlar["rating"].value_counts().sort_index()

renkler = [MAVI if p == int(p) else TURUNCU for p in dagilim.index]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(dagilim.index.astype(str), dagilim.values, color=renkler)
ax.set_xlabel("Puan (yıldız)")
ax.set_ylabel("Puan sayısı")
ax.set_title("Puanların dağılımı (mavi: tam yıldız, turuncu: buçuklu)")
for i, v in enumerate(dagilim.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

print("Ortalama:", round(puanlar["rating"].mean(), 2))
print("Medyan  :", puanlar["rating"].median())

**🔮 Düşün:** Dağılım simetrik değil: puanlar sağa (yüksek puanlara) yığılmış. Neden? İnsanlar rastgele film mi izliyor, yoksa beğeneceğini tahmin ettiği filmleri mi seçiyor?

*Cevabım:* ...

> Bu durum öneri sistemlerinde **seçim yanlılığı** (selection bias) olarak bilinir: kullanıcı kötü bulacağını düşündüğü filmi zaten izlemez, puanlamaz. Puansız bir hücre "beğenmedi" demek **değildir**.

**🔧 Değiştir:** Yukarıdaki grafiği **yüzde** olarak çiz: her çubuğun yüksekliği o puanın bütün puanlar içindeki payı olsun. (İpucu: `value_counts(normalize=True)`.)

In [ ]:
# Kodunu buraya yaz

---
## Bölüm 3: Uzun kuyruk: kim çok puanlıyor, hangi film çok puanlanıyor? (20 dk)
### 3.1 Kullanıcı başına puan sayısı
**🔮 Tahmin et:** Bir kullanıcı ortalama kaç film puanlamıştır? En az puan veren kullanıcı kaç puan vermiştir?

*Tahminim:* ...

In [ ]:
kullanici_sayisi = puanlar.groupby("userId").size()
kullanici_sayisi.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(kullanici_sayisi, bins=50, color=MAVI)
axes[0].set_xlabel("Kullanıcının verdiği puan sayısı")
axes[0].set_ylabel("Kullanıcı sayısı")
axes[0].set_title("Doğrusal ölçek")

axes[1].hist(kullanici_sayisi, bins=np.logspace(np.log10(20), np.log10(3000), 40), color=MAVI)
axes[1].set_xscale("log")
axes[1].axvline(kullanici_sayisi.median(), color=KIRMIZI, linestyle="--",
                label=f"medyan = {kullanici_sayisi.median():.0f}")
axes[1].set_xlabel("Kullanıcının verdiği puan sayısı (log ölçek)")
axes[1].set_title("Logaritmik ölçek")
axes[1].legend()

plt.tight_layout()
plt.show()

**🔮 Düşün:** Kimse 20'den az puan vermemiş. Bu bir rastlantı mı? `README.txt` dosyasında bu konuda bir cümle ara.

*Cevabım:* ...

<details><summary>💡 İpucu 1</summary>

README: *"All selected users had rated at least 20 movies."* Veri seti hazırlanırken **seçilmiş**. Gerçek bir sistemde çok az puan veren (soğuk başlangıç, *cold start*) kullanıcılar da olur.

</details>

### 3.2 Film başına puan sayısı
**🔮 Tahmin et:** Puan almış filmlerin yüzde kaçı **yalnızca bir kez** puanlanmıştır? (%5? %15? %35?)

*Tahminim:* ...

In [ ]:
film_sayisi = puanlar.groupby("movieId").size()
print("Medyan puan sayısı      :", film_sayisi.median())
print("Tek puanlı film sayısı  :", (film_sayisi == 1).sum())
print("Tek puanlı filmlerin payı: %", round(100 * (film_sayisi == 1).mean(), 1))

Şimdi filmleri en çok puanlanandan en aza doğru sıralayıp çizelim. Bu grafiğe **uzun kuyruk** (long tail) denir.

In [ ]:
sirali = film_sayisi.sort_values(ascending=False).reset_index(drop=True)
ilk_yuzde20 = int(len(sirali) * 0.20)
pay = sirali.iloc[:ilk_yuzde20].sum() / sirali.sum()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.fill_between(sirali.index[:ilk_yuzde20], sirali.values[:ilk_yuzde20], color=KIRMIZI, alpha=0.8,
                label=f"Popüler %20 film: puanların %{100 * pay:.0f}'i")
ax.fill_between(sirali.index[ilk_yuzde20:], sirali.values[ilk_yuzde20:], color=GRI, alpha=0.8,
                label=f"Kalan %80 film (uzun kuyruk): puanların %{100 * (1 - pay):.0f}'i")
ax.set_xlabel("Film sırası (en çok puanlanandan en aza)")
ax.set_ylabel("Puan sayısı")
ax.set_title("Uzun kuyruk: az sayıda film, puanların çoğunu topluyor")
ax.legend()
plt.tight_layout()
plt.show()

**🔮 Düşün:** H06'da Online Retail'de de benzer bir şey görmüştük: birkaç ürün çok sık satılıyordu. Bir öneri sistemi yalnızca kırmızı bölgedeki filmleri önerirse ne olur? Kullanıcıya **yeni** bir şey söylemiş olur mu?

*Cevabım:* ...

**✍️ Kendin yaz:** En çok puanlanan 10 filmin **adlarını** ve puan sayılarını yatay çubuk grafikle çiz. (`film_sayisi` yalnızca `movieId` biliyor; adları `filmler` tablosundan almalısın.)

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

`film_sayisi.nlargest(10)` sana 10 `movieId` ve sayı verir. Adları eklemek için `filmler.set_index("movieId")["title"]` ile bir eşleme kur ve `.map()` kullan, ya da `pd.merge` ile birleştir.

</details>

<details><summary>💡 İpucu 2</summary>

```python
ad = filmler.set_index("movieId")["title"]
on = film_sayisi.nlargest(10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(on.index.map(ad), on.values, color=MAVI)
ax.invert_yaxis()
```

</details>

### 3.3 Kullanıcı × film tablosu ne kadar dolu?
Öneri sistemlerinin temel tablosu: satırlar kullanıcı, sütunlar film, hücreler puan. H06'daki **sepet tablosunun** puanlı hâli.

**🔮 Tahmin et:** 610 kullanıcı × 9.724 film = yaklaşık 5,9 milyon hücre var. Bunların yüzde kaçı dolu?

*Tahminim:* ...

In [ ]:
hucre = n_kullanici * n_film
doluluk = len(puanlar) / hucre
print(f"Toplam hücre : {hucre:,}")
print(f"Dolu hücre   : {len(puanlar):,}")
print(f"Doluluk      : %{100 * doluluk:.2f}")
print(f"Seyreklik    : %{100 * (1 - doluluk):.2f}")

Tablonun bir köşesini görelim. Önce **en çok puan veren** 60 kullanıcı ve **en çok puanlanan** 60 film (en dolu köşe), sonra **rastgele** 60 kullanıcı ve 60 film.

In [ ]:
# 1) Kullanıcı × film tablosu (pivot): puan yoksa NaN
matris = puanlar.pivot_table(index="userId", columns="movieId", values="rating")

# 2) En dolu köşe ve rastgele bir köşe
dolu_kose = matris.loc[kullanici_sayisi.nlargest(60).index, film_sayisi.nlargest(60).index]
rastgele_kose = matris.sample(60, random_state=1).sample(60, axis=1, random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, kose, baslik in [(axes[0], dolu_kose, "En aktif kullanıcılar × en popüler filmler"),
                         (axes[1], rastgele_kose, "Rastgele kullanıcılar × rastgele filmler")]:
    resim = ax.imshow(kose.values, cmap="viridis", vmin=0.5, vmax=5, aspect="auto", interpolation="nearest")
    ax.set_title(f"{baslik}\nhücrelerin %{100 * kose.notna().values.mean():.1f}'i dolu", fontsize=10)
    ax.set_xlabel("Film")
    ax.set_ylabel("Kullanıcı")
fig.colorbar(resim, ax=axes, label="Puan (beyaz: puan yok)")
plt.show()

**🔮 Düşün:** Sağdaki grafik neredeyse bomboş. Öneri sistemi tam da bu boş hücreleri **tahmin etmeye** çalışır. Bu kadar seyrek bir tablodan iki kullanıcının "benzer" olduğunu nasıl anlarız? İki rastgele kullanıcının ortak puanladığı film sayısı kaç olabilir?

*Cevabım:* ...

---
## Bölüm 4: Türler (15 dk)
`genres` sütunu birden çok türü `|` ile ayırarak tek bir metinde tutuyor: `Adventure|Animation|Children|Comedy|Fantasy`.

**🔮 Tahmin et:** `filmler["genres"].nunique()` kaç çıkar? 20 civarı mı, yoksa çok daha fazla mı?

*Tahminim:* ...

In [ ]:
filmler["genres"].nunique()

Yüzlerce "tür" çıktı, çünkü pandas her **kombinasyonu** ayrı bir değer sayıyor. Türleri saymak için önce metni parçalayıp (`split`), sonra her parçayı ayrı bir satıra açmamız (`explode`) gerekir:

In [ ]:
ornek = filmler.head(3)[["title", "genres"]].copy()
ornek["tur"] = ornek["genres"].str.split("|")
print(ornek[["title", "tur"]])
print()
print(ornek[["title", "tur"]].explode("tur"))

**🔮 Tahmin et:** `explode` sonrası 3 film kaç satır oldu? Neden?

*Tahminim:* ...

In [ ]:
turler = filmler.assign(tur=filmler["genres"].str.split("|")).explode("tur")
tur_sayisi = turler["tur"].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
renk = [KIRMIZI if t == "(no genres listed)" else MAVI for t in tur_sayisi.index]
ax.barh(tur_sayisi.index, tur_sayisi.values, color=renk)
ax.invert_yaxis()
ax.set_xlabel("Film sayısı")
ax.set_title("Türlere göre film sayısı (bir film birden çok türde olabilir)")
for i, v in enumerate(tur_sayisi.values):
    ax.text(v, i, f" {v:,}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

**🔮 Düşün:** Kırmızı çubuk bir tür mü? Bu filmlerle ne yapmalıyız? (Bölüm 6'da karar vereceğiz.)

**Bir film kaç türde?**

In [ ]:
tur_adedi = filmler["genres"].str.split("|").str.len()
tur_adedi.value_counts().sort_index().plot.bar(color=MAVI, figsize=(7, 3), rot=0,
                                               xlabel="Filmin tür sayısı", ylabel="Film sayısı",
                                               title="Bir film kaç türde etiketlenmiş?")
plt.tight_layout()
plt.show()

### 4.1 Hangi tür daha çok beğeniliyor?
Puanları türlerle birleştirelim. `puanlar` her satırda bir puan, `turler` her satırda bir (film, tür) çifti tutuyor; `movieId` üzerinden **birleştir** (`merge`).

**🔮 Tahmin et:** Ortalama puanı en yüksek tür hangisi olur? En düşük hangisi?

*Tahminim:* ...

In [ ]:
puan_tur = puanlar.merge(turler[["movieId", "tur"]], on="movieId")
print("Birleşmeden önce:", len(puanlar), " sonra:", len(puan_tur))

tur_ozet = (puan_tur.groupby("tur")["rating"]
            .agg(ortalama="mean", puan_sayisi="size")
            .query("puan_sayisi >= 100")
            .sort_values("ortalama"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(tur_ozet.index, tur_ozet["ortalama"], color=MAVI)
ax.axvline(puanlar["rating"].mean(), color=KIRMIZI, linestyle="--", label="Genel ortalama")
ax.set_xlim(3, 4.1)
ax.set_xlabel("Ortalama puan (x ekseni 3'ten başlıyor!)")
ax.set_title("Türlere göre ortalama puan")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**🔮 Düşün:**
1. Birleşmeden sonra satır sayısı neden **arttı**? Bir puan birden çok kez sayılmış olabilir mi?
2. x ekseni 3'ten başlıyor. Eksen 0'dan başlasaydı türler arasındaki fark nasıl görünürdü? Hangisi daha dürüst bir grafik?
3. `query("puan_sayisi >= 100")` satırı neden var? Silsek hangi tür listeye girer ve sonucu nasıl etkiler?

*Cevaplarım:* ...

**🔧 Değiştir:** Grafiği `ax.set_xlim(0, 5)` ile yeniden çiz ve iki grafiği karşılaştır.

In [ ]:
# Kodunu buraya yaz

---
## Bölüm 5: Zaman (15 dk)
İki farklı zaman var:
- **Filmin yılı:** başlığın sonunda parantez içinde: `Toy Story (1995)`
- **Puanın zamanı:** `timestamp` sütununda, saniye cinsinden

### 5.1 Başlıktan yıl çıkarmak
Başlığın sonundaki `(dört rakam)` kısmını bir **düzenli ifade** (regular expression) ile çekeceğiz:

| Parça | Anlamı |
|---|---|
| `\(` | açılış parantezi (parantez regex'te özel karakter, başına `\` koyarız) |
| `(\d{4})` | dört rakam; dıştaki parantez "bu kısmı bana ver" demek |
| `\)` | kapanış parantezi |
| `$` | metnin sonu |

**🐞 Hata avı:** Aşağıdaki kod yılı çıkarıyor ama bazı filmlerde boş (`NaN`) dönüyor. Hücreyi çalıştır ve yılı çıkmayan başlıklara bak. Hepsinde gerçekten yıl yok mu?

In [ ]:
yil = filmler["title"].str.extract(r"\((\d{4})\)$")[0]
print("Yılı çıkmayan film sayısı:", yil.isna().sum())
filmler.loc[yil.isna(), "title"].tolist()

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

Listede `'Toy Story (1995) '` gibi sonu **boşlukla** biten başlıkları ara. `$` "metnin sonu" demek; boşluk varsa sondaki karakter `)` olmuyor. `repr()` ile yazdırırsan boşluğu görürsün.

</details>

<details><summary>💡 İpucu 2</summary>

Önce `.str.strip()` ile baştaki/sondaki boşlukları at, sonra `extract` uygula: `filmler["title"].str.strip().str.extract(...)`. Kalan 13 filmin başlığında gerçekten yıl yok.

</details>

Düzeltilmiş hâliyle yılı tabloya ekleyip filmlerin yıllara dağılımına bakalım:

In [ ]:
filmler["yil"] = filmler["title"].str.strip().str.extract(r"\((\d{4})\)$")[0].astype("Int64")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(filmler["yil"].dropna(), bins=range(1900, 2021, 2), color=MAVI)
ax.set_xlabel("Filmin yılı")
ax.set_ylabel("Film sayısı")
ax.set_title("Veri setindeki filmler hangi yıllardan?")
plt.tight_layout()
plt.show()

### 5.2 Puanlar ne zaman verilmiş?
`pd.to_datetime(..., unit="s")` saniye sayısını tarihe çevirir.

In [ ]:
puanlar["zaman"] = pd.to_datetime(puanlar["timestamp"], unit="s")
print("İlk puan:", puanlar["zaman"].min())
print("Son puan:", puanlar["zaman"].max())

yillik = puanlar.groupby(puanlar["zaman"].dt.year).agg(puan_sayisi=("rating", "size"),
                                                       ortalama=("rating", "mean"))

fig, ax1 = plt.subplots(figsize=(10, 3.8))
ax1.bar(yillik.index, yillik["puan_sayisi"], color=GRI)
ax1.set_xlabel("Puanın verildiği yıl")
ax1.set_ylabel("Puan sayısı", color=GRI)
ax2 = ax1.twinx()
ax2.plot(yillik.index, yillik["ortalama"], color=KIRMIZI, marker="o")
ax2.set_ylabel("Ortalama puan", color=KIRMIZI)
ax2.set_ylim(2.5, 4.5)
ax1.set_title("Yıllara göre puan sayısı (çubuk) ve ortalama puan (çizgi)")
plt.tight_layout()
plt.show()

**🔮 Düşün:** Bazı yıllarda puan sayısı birden fırlıyor. 610 kullanıcının "hep birlikte" daha çok film izlemesi mi, yoksa birkaç çok aktif kullanıcının o yıl sisteme katılması mı? Nasıl kontrol edersin?

*Cevabım:* ...

**✍️ Kendin yaz:** Her yıl için **kaç farklı kullanıcının** puan verdiğini bul ve çubuk grafikle çiz. (İpucu: `groupby(...)["userId"].nunique()`)

In [ ]:
# Kodunu buraya yaz

**🕵️ İz sür:** Bir film, çıkış yılından **önce** puanlanabilir mi? Puanın yılı filmin yılından küçük olan satırları bul. Kaç tane? Hangi filmler?

In [ ]:
# Kodunu buraya yaz

<details><summary>💡 İpucu 1</summary>

`puanlar` ile `filmler[["movieId", "title", "yil"]]` tablosunu `movieId` üzerinden birleştir; sonra `zaman.dt.year < yil` maskesiyle süz.

</details>

<details><summary>💡 İpucu 2</summary>

3 satır çıkar. Film festivalde gösterilmiş, vizyona bir yıl sonra girmiş olabilir; ya da başlıktaki yıl vizyon yılıdır. **Veri hatası mı, gerçek mi?** Çok az satır olduğu için silmeye değmez; ama böyle bir kontrolü yapmış olmak önemli.

</details>

---
## Bölüm 6: Veri temizleme (30 dk)
Bölüm 1'de `isna()` ve `duplicated()` hiçbir sorun göstermemişti. Ama yol boyunca şunlara rastladık:

| # | Sorun | Nerede gördük? |
|---|---|---|
| 1 | Başlıkların sonunda boşluk | Bölüm 5.1 🐞 |
| 2 | `(no genres listed)` bir tür gibi sayılıyor | Bölüm 4 |
| 3 | Az puanlı filmlerin ortalaması güvenilmez | (şimdi göreceğiz) |
| 4 | Aynı film iki farklı `movieId` ile | (şimdi göreceğiz) |
| 5 | Etiketlerde büyük/küçük harf tutarsızlığı | (şimdi göreceğiz) |
| 6 | Puansız filmler | Bölüm 1.2 🕵️ |

Bunlar **anlam** sorunlarıdır: `isna()` görmez, çünkü hücre boş değildir; `duplicated()` görmez, çünkü satırlar birebir aynı değildir.

### 6.1 "En iyi filmler" listesi
**🔮 Tahmin et:** Ortalama puanı en yüksek 10 film hangileri olur? Aklına gelen birkaç klasik filmi yaz.

*Tahminim:* ...

In [ ]:
film_ozet = (puanlar.groupby("movieId")["rating"]
             .agg(ortalama="mean", puan_sayisi="size")
             .join(filmler.set_index("movieId")["title"]))
film_ozet.sort_values("ortalama", ascending=False).head(10)

**🔮 Düşün:** Listedeki filmleri tanıyor musun? `puan_sayisi` sütununa bak. Tek bir kişinin 5 yıldızı, 300 kişinin 4,4 ortalamasından "daha iyi" mi?

Bir grafikle görelim:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(film_ozet["puan_sayisi"], film_ozet["ortalama"], s=6, alpha=0.3, color=MAVI)
ax.set_xscale("log")
ax.axhline(puanlar["rating"].mean(), color=GRI, linestyle="--", label="Genel ortalama")
ax.axvline(30, color=KIRMIZI, linestyle="--", label="30 puan eşiği")
ax.set_xlabel("Filmin aldığı puan sayısı (log ölçek)")
ax.set_ylabel("Filmin ortalama puanı")
ax.set_title("Her nokta bir film: az puanlı filmlerin ortalaması uçlara dağılıyor")
ax.legend()
plt.tight_layout()
plt.show()

Grafik bir **huni** gibi: solda (az puan) ortalamalar 0,5 ile 5 arasında her yere dağılmış, sağa gittikçe (çok puan) genel ortalamaya yaklaşıyor. Az gözlemden hesaplanan ortalama **güvenilmezdir**. Bu, istatistikteki *büyük sayılar yasası*nın görsel hâlidir.

**Karar:** Filmleri silmiyoruz (öneri sistemi uzun kuyruktaki filmleri de bilmeli), ama **sıralama yaparken** en az 30 puan şartı koyuyoruz.

In [ ]:
ESIK = 30
film_ozet[film_ozet["puan_sayisi"] >= ESIK].sort_values("ortalama", ascending=False).head(10)

**🔧 Değiştir:** Eşiği 10, 50 ve 100 yap. Listenin başı nasıl değişiyor? Kaç film listeye girme hakkı kazanıyor?

In [ ]:
# Kodunu buraya yaz

### 6.2 Aynı adlı filmler
**🕵️ İz sür:** Aynı başlık birden fazla kez geçiyor mu?

In [ ]:
ayni_ad = filmler[filmler["title"].duplicated(keep=False)].sort_values("title")
ayni_ad["puan_sayisi"] = ayni_ad["movieId"].map(film_sayisi).fillna(0).astype(int)
ayni_ad[["movieId", "title", "genres", "puan_sayisi"]]

**🔮 Düşün:** Bunlar gerçekten farklı filmler mi (aynı yıl, aynı adla iki film çekilmiş olabilir), yoksa aynı filmin iki kaydı mı? `genres` sütununa ve `baglar` tablosundaki `imdbId`'lere bakarak karar ver.

In [ ]:
ayni_ad.merge(baglar, on="movieId")[["movieId", "title", "imdbId", "tmdbId"]]

<details><summary>💡 Yorum</summary>

`imdbId`'ler **farklı**. Yani IMDb'ye göre bunlar ayrı kayıtlar; bazıları gerçekten farklı yapımlar olabilir (ör. aynı adlı TV filmi). Ama bir **öneri listesinde** kullanıcıya iki tane "Emma (1996)" göstermek kafa karıştırır. Ayrıca film adını anahtar olarak kullanacağımız yerlerde (H09 içerik tabanlı defter) aynı ad iki satıra karşılık gelir.

**Sınıf kararı:** Aynı adlı kayıtlardan **en çok puan alanı** tutuyoruz; diğerinin puanlarını tutulan kayda aktarıyoruz.

</details>

### 6.3 Etiketler
**🔮 Tahmin et:** `etiketler["tag"]` sütununda kaç farklı etiket var? Hepsini küçük harfe çevirirsek bu sayı değişir mi?

*Tahminim:* ...

In [ ]:
print("Farklı etiket                 :", etiketler["tag"].nunique())
print("Küçük harfe çevirince         :", etiketler["tag"].str.lower().nunique())
print("Küçük harf + boşluk atınca    :", etiketler["tag"].str.lower().str.strip().nunique())

# Aynı etiketin farklı yazılışları
yazilis = etiketler.assign(kucuk=etiketler["tag"].str.lower().str.strip()).groupby("kucuk")["tag"].unique()
yazilis[yazilis.str.len() > 1].head(8)

In [ ]:
en_cok = etiketler["tag"].str.lower().str.strip().value_counts().head(15)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(en_cok.index, en_cok.values, color=MOR)
ax.invert_yaxis()
ax.set_xlabel("Kullanım sayısı")
ax.set_title("En çok kullanılan 15 etiket (küçük harfe çevrilmiş)")
plt.tight_layout()
plt.show()

**🔮 Düşün:** En çok kullanılan etiket `in netflix queue`. Bu etiket filmin **içeriği** hakkında bir şey söylüyor mu? İçerik tabanlı bir öneri sisteminde (H09) bu etiketi kullanmak mantıklı mı?

*Cevabım:* ...

### 6.4 Temizliği uygula
Kararlarımızı tek yerde, sırayla uyguluyoruz. Ham tablolara dokunmuyoruz: temiz hâller `f`, `p`, `e` adlı yeni değişkenlerde.

In [ ]:
f = filmler.copy()
p = puanlar.copy()
e = etiketler.copy()

# Karar 1: başlıkların baş/son boşluklarını at
f["title"] = f["title"].str.strip()

# Karar 2: "(no genres listed)" bir tür değil: eksik değer olarak işaretle
print("Türsüz film:", (f["genres"] == "(no genres listed)").sum())
f["genres"] = f["genres"].replace("(no genres listed)", pd.NA)

# Karar 3: aynı adlı filmlerde en çok puanlananı tut, diğerinin puanlarını ona aktar
f["puan_sayisi"] = f["movieId"].map(film_sayisi).fillna(0).astype(int)
f = f.sort_values("puan_sayisi", ascending=False)
tutulan = f.drop_duplicates("title").set_index("title")["movieId"]       # ad → tutulan movieId
eslesme = dict(zip(f["movieId"], f["title"].map(tutulan)))              # her movieId → tutulan movieId
p["movieId"] = p["movieId"].map(eslesme)
e["movieId"] = e["movieId"].map(eslesme)
f = adim(f, ~f["title"].duplicated(), "Aynı adlı film kaydı")

# Aktarma sonrası aynı kullanıcı aynı filme iki puan vermiş olabilir: en son vereni tut
p = p.sort_values("zaman")
p = adim(p, ~p.duplicated(["userId", "movieId"], keep="last"), "Aynı kullanıcı-film, eski puan")

# Karar 4: etiketleri küçük harf ve boşluksuz yap
e["tag"] = e["tag"].str.lower().str.strip()

# Karar 5: puan sayısını yeniden hesapla (aktarmadan sonra değişti)
f["puan_sayisi"] = f["movieId"].map(p.groupby("movieId").size()).fillna(0).astype(int)
f = f.sort_values("movieId").reset_index(drop=True)
p = p.reset_index(drop=True)

print()
kontrol_temiz(f, p, e)

**🔮 Düşün:** Puan almamış filmleri (`puan_sayisi == 0`) **silmedik**. Neden? Hangi tür bir öneri sistemi bu filmleri yine de önerebilir? (İpucu: H09 içerik tabanlı defter.)

*Cevabım:* ...

**🐞 Hata avı:** Aşağıdaki hücre temiz tabloda her filmin kaç türde olduğunu tam sayı olarak hesaplamaya çalışıyor (Bölüm 4'teki kodun aynısı) ama hata veriyor. Hata mesajının **son satırını** oku. Ham tabloda çalışan kod temiz tabloda neden çalışmıyor?

In [ ]:
f["tur_adedi"] = f["genres"].str.split("|").str.len().astype(int)
f["tur_adedi"].value_counts()

In [ ]:
# Düzeltilmiş kod

<details><summary>💡 İpucu 1</summary>

Karar 2'de `(no genres listed)` yerine `NA` koyduk. `NA` bir metin değildir: `.str.split()` ve `.str.len()` ona dokunmaz, sonuç yine `NA` olur. `NA` bir `int`'e çevrilemez.

</details>

<details><summary>💡 İpucu 2</summary>

İki seçenek: türsüz filmin tür sayısını 0 kabul et (`.fillna(0).astype(int)`) ya da eksik değeri taşıyabilen tam sayı tipini kullan (`.astype("Int64")`, büyük **I** ile). Hangisi daha doğru? Türsüz film "0 türde" mi, yoksa "türü bilinmiyor" mu?

</details>

### 6.5 Önce ve sonra

In [ ]:
karsilastirma = pd.DataFrame({
    "Ham":   [len(filmler), len(puanlar), filmler["title"].duplicated().sum(),
              (filmler["genres"] == "(no genres listed)").sum(), etiketler["tag"].nunique()],
    "Temiz": [len(f), len(p), f["title"].duplicated().sum(),
              f["genres"].isna().sum(), e["tag"].nunique()],
}, index=["Film sayısı", "Puan sayısı", "Tekrarlı başlık", "Türsüz film", "Farklı etiket"])
karsilastirma

### 6.6 Temiz veriyi kaydet
Sonraki haftalarda (işbirlikçi filtreleme, matris çarpanlarına ayırma) bu temiz dosyaları okuyacağız. Dosyalar `Veri/movielens/` altına yazılır ve Git'e girmez.

In [ ]:
f.to_csv(os.path.join("Veri", "movielens", "filmler_temiz.csv"), index=False)
p.to_csv(os.path.join("Veri", "movielens", "puanlar_temiz.csv"), index=False)
e.to_csv(os.path.join("Veri", "movielens", "etiketler_temiz.csv"), index=False)
print("Kaydedildi:", os.listdir(os.path.join("Veri", "movielens")))

---
## Bölüm 7: Online Retail ile MovieLens (5 dk)
**✍️ Tabloyu doldur:** Bu dönem iki veri setiyle çalışıyoruz. Bugün gördüklerine göre karşılaştır:

| | Online Retail II | MovieLens |
|---|---|---|
| Bir satır ne anlatıyor? | Bir faturadaki bir ürün | |
| "Kullanıcı" kim? | Müşteri (`Customer ID`) | |
| "Öğe" ne? | Ürün (`StockCode`) | |
| Tercih nasıl ölçülüyor? | Satın aldı / almadı (**örtük**, implicit) | |
| Tablonun ne kadarı dolu? | (H06'daki sepet tablosu) | |
| Uzun kuyruk var mı? | Evet | |
| En önemli temizlik sorunu | İptaller, negatif adetler, boş müşteri | |

**🔮 Düşün:** Online Retail'de "müşteri bu ürünü almadı" bilgisi ile MovieLens'te "kullanıcı bu filmi puanlamadı" bilgisi aynı şeyi mi söyler?

---
#### 📘 Özet
- MovieLens **dört tablodan** oluşur; tablolar `movieId` anahtarıyla birleşir (`merge`). Kullanıcıları yalnızca puanlarından tanırız.
- Puanlar **açık geri bildirimdir** (explicit) ama yüksek puanlara yığılır: insanlar beğeneceğini düşündüğü filmi izler (**seçim yanlılığı**).
- Hem kullanıcılarda hem filmlerde **uzun kuyruk** vardır: az sayıda film puanların çoğunu toplar; filmlerin üçte biri yalnızca bir kez puanlanmıştır.
- Kullanıcı × film tablosu **%98'den fazla boştur** (seyreklik). Öneri sistemleri bu boş hücreleri tahmin etmeye çalışır.
- Çok değerli metin sütunları (`genres`) **`split` + `explode`** ile satırlara açılır.
- Zaman: başlıktan yıl **regex** ile, `timestamp` ise **`pd.to_datetime(..., unit="s")`** ile çıkar.
- `isna()` ve `duplicated()` her sorunu görmez. **Anlam sorunları** (sondaki boşluk, sahte kategori, aynı ada iki kayıt, büyük/küçük harf, az gözlemli ortalama) ancak veriye bakarak bulunur.
- Az gözlemden hesaplanan ortalama güvenilmezdir; sıralamada **en az puan sayısı** eşiği kullanılır.

| Araç | Ne işe yarar? | Örnek |
|---|---|---|
| `.nunique()` | Farklı değer sayısı | `puanlar["userId"].nunique()` |
| `.isin()` | Değer başka bir listede var mı? | `filmler["movieId"].isin(puanlar["movieId"])` |
| `.str.split("|")` + `.explode()` | Çok değerli hücreyi satırlara açar | türler |
| `.str.extract(regex)` | Metinden kalıba uyan parçayı çeker | başlıktan yıl |
| `pd.to_datetime(x, unit="s")` | Unix zamanını tarihe çevirir | `timestamp` |
| `.pivot_table()` | Uzun tabloyu kullanıcı × film tablosuna çevirir | seyreklik |
| `.merge(..., on=)` | İki tabloyu ortak sütundan birleştirir | puan + tür |

---
### 🏠 Ev alıştırmaları
1. **✍️ Kullanıcı profili:** `userId == 414` (en çok puan veren kullanıcı) en çok hangi türlerde film puanlamış? Ortalama puanı genel ortalamadan yüksek mi, düşük mü? Bir grafikle göster.
2. **🕵️ Cömert ve cimri kullanıcılar:** Her kullanıcının ortalama puanını hesapla ve histogramını çiz. Bütün filmlere aynı puanı veren kullanıcı var mı? (İpucu: kullanıcı başına puanların standart sapması.) Böyle bir kullanıcı öneri sistemine ne bilgi verir?
3. **✍️ Tür × yıl:** Her on yıl (1950'ler, 1960'lar, ...) için en çok film hangi türde çekilmiş? `yil // 10 * 10` ile on yılı bul, `pd.crosstab` ile tür × on yıl tablosu kur ve ısı haritası (`imshow`) olarak çiz.
4. **🔧 Başlık düzeltme:** Pek çok başlık `"American President, The (1995)"` biçiminde yazılmış. Bunları `"The American President (1995)"` biçimine çeviren bir fonksiyon yaz. Kaç başlık etkileniyor? (`, The`, `, A`, `, An` kalıplarına bak.)